# Initial question

How can Jaffle Shop get more repeat customers using deals or discounts?

The recommendation is a two-step drink coupon: a free drink on the next visit, followed by a free drink with any jaffle on a third visit. This encourages the drink-focused habits seen among repeat customers in our sample. The offer should be tested to see whether it creates more regular customers.


# How I approached the question

I used dbt to turn the raw BigQuery data into customer and order models, then used Polars and Lets-Plot to analyze them. `dim__customers` summarizes each customer, while `fct__orders` includes spending, product quantities, and visit history. Key and business-rule tests check the models before analysis. One order counts as one visit, and a repeat customer has at least three visits in the full sample.

This workshop sample covers September 1-16, 2016, with 128 customers and 686 orders from Philadelphia. Customers who first appear later have less time to return, so one observed visit does not mean a customer is permanently lost. Group labels use the full sample, including visits that happened after a customer's first order; these comparisons describe observed patterns, not predictions or proof that discounts work.


The notebook reads the two marts and the product/store reference models from BigQuery using the existing dbt profile. Credentials stay in the local `.dbt` folder. Use the **Python (Jaffle Shop)** kernel in WSL to rerun the notebook; the saved HTML can be viewed without a database connection. Results below were checked against the workshop seed data on November 16, 2026.


In [1]:
from pathlib import Path
from decimal import Decimal
import os
import yaml
import polars as pl
from google.cloud import bigquery
from google.oauth2 import service_account
from IPython.display import display
from lets_plot import *

LetsPlot.setup_html(no_js=True)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_float_precision(2)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

profile = yaml.safe_load((Path.home() / ".dbt/profiles.yml").read_text())["university_workshop"]
config = profile["outputs"][profile["target"]]
key_path = Path(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", Path.home() / ".dbt/bigquery-key.json"))
credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(
    project=config["project"],
    credentials=credentials,
    location=config.get("location", "US"),
)

def read_model(model):
    query = f"select * from `{config['project']}.{config['dataset']}.{model}`"
    job = client.query(query, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000))
    rows = [
        {key: float(value) if isinstance(value, Decimal) else value for key, value in row.items()}
        for row in job.result(timeout=90)
    ]
    return pl.DataFrame(rows, infer_schema_length=None)

customers = read_model("dim__customers")
orders = read_model("fct__orders")
products = read_model("stg__jaffle_shop__raw_products").with_columns(
    pl.col("product_name").str.strip_chars()
)
stores = read_model("stg__jaffle_shop__raw_stores")
client.close()

repeat_customers = customers.filter(pl.col("is_repeat_customer"))
active_customers = customers.filter(pl.col("total_visits") > 0)
group_order = ["1", "2", "3+"]
group_labels = ["1 visit", "2 visits", "3+ visits"]
group_colors = ["#64748B", "#3B82F6", "#159A88"]
blue = "#3B82F6"
green = "#159A88"

sample_summary = pl.DataFrame({
    "Measure": ["Sample begins", "Sample ends", "Customers with orders", "Customers without orders", "Orders", "Purchased items", "Sales including tax", "Stores with orders"],
    "Value": [
        str(orders["order_date"].min()),
        str(orders["order_date"].max()),
        f"{active_customers.height:,}",
        str(customers.filter(pl.col("total_visits") == 0).height),
        f"{orders.height:,}",
        f"{orders['item_count'].sum():,}",
        f"${orders['order_total_dollars'].sum():,.2f}",
        str(orders["store_id"].n_unique()),
    ],
})
display(sample_summary)


Measure,Value
"""Sample begins""","""2016-09-01"""
"""Sample ends""","""2016-09-16"""
"""Customers with orders""","""128"""
"""Customers without orders""","""0"""
"""Orders""","""686"""
"""Purchased items""","""997"""
"""Sales including tax""","""$7,226.67"""
"""Stores with orders""","""1"""


# Insights

The six questions below compare customers with one visit, two visits, and three or more visits. Percentages use customer counts unless a chart explicitly refers to orders or purchased items.


## Who are repeat customers?

Repeat customers are the 105 people with at least three orders in this sample. They made a median of seven visits and an average of 6.19 visits, with counts ranging from three to ten. They placed 650 of the 686 orders and generated 86.5% of sales, making frequent visits an important part of this sample's sales.


In [2]:
repeat_orders = orders.filter(pl.col("is_repeat_customer"))
repeat_profile = pl.DataFrame({
    "Measure": ["Repeat customers", "Share of customers", "Median visits per repeat customer", "Average visits per repeat customer", "Visit range", "Orders from repeat customers", "Share of all orders", "Sales including tax", "Share of all sales"],
    "Value": [
        str(repeat_customers.height),
        f"{100 * repeat_customers.height / active_customers.height:.1f}%",
        f"{repeat_customers['total_visits'].median():.0f}",
        f"{repeat_customers['total_visits'].mean():.2f}",
        f"{repeat_customers['total_visits'].min()} to {repeat_customers['total_visits'].max()}",
        str(repeat_orders.height),
        f"{100 * repeat_orders.height / orders.height:.1f}%",
        f"${repeat_orders['order_total_dollars'].sum():,.2f}",
        f"{100 * repeat_orders['order_total_dollars'].sum() / orders['order_total_dollars'].sum():.1f}%",
    ],
})
display(repeat_profile)


Measure,Value
"""Repeat customers""","""105"""
"""Share of customers""","""82.0%"""
"""Median visits per repeat customer""","""7"""
"""Average visits per repeat customer""","""6.19"""
"""Visit range""","""3 to 10"""
"""Orders from repeat customers""","""650"""
"""Share of all orders""","""94.8%"""
"""Sales including tax""","""$6,249.39"""
"""Share of all sales""","""86.5%"""


## What percentage of customers have 1, 2, or 3+ visits?

Of the 128 customers, 10 had one visit (7.8%), 13 had two visits (10.2%), and 105 had three or more visits (82.0%). The 23 customers with one or two visits are a possible audience for a return offer. This is the share of customers in a short sample, not a long-term retention rate, and the smaller groups give us less evidence about their habits.


In [3]:
visit_groups = (
    active_customers.group_by("visit_group")
    .agg(pl.len().alias("customers"))
    .with_columns((100 * pl.col("customers") / active_customers.height).alias("percent"))
    .sort("visit_group")
    .with_columns(
        (pl.col("percent").round(1).cast(pl.String) + "% (n=" + pl.col("customers").cast(pl.String) + ")").alias("label")
    )
)
display(visit_groups.select(
    pl.col("visit_group").alias("Visits"),
    pl.col("customers").alias("Customers"),
    pl.col("percent").round(1).alias("Percent of customers"),
))
visit_share_plot = (
    ggplot(visit_groups, aes(x="visit_group", y="percent", fill="visit_group"))
    + geom_bar(stat="identity", width=0.6, show_legend=False)
    + geom_text(aes(label="label"), nudge_y=3, size=11, show_legend=False)
    + scale_fill_manual(values=group_colors, breaks=group_order)
    + scale_x_discrete(limits=group_order, labels=group_labels)
    + scale_y_continuous(limits=[0, 100], breaks=[0, 20, 40, 60, 80, 100])
    + labs(title="Most observed customers made at least three visits", subtitle="128 customers; percentages count customers, not orders", x="", y="Customers (%)")
    + theme_minimal()
    + ggsize(850, 430)
)
display(visit_share_plot)


Visits,Customers,Percent of customers
"""1""",10,7.80
"""2""",13,10.20
"""3+""",105,82.00


## How long do repeat customers take to place a second order?

Among the 105 repeat customers, the median wait from the first order to the second was 2.98 days, and the average was 3.20 days. Just over half (53.3%) returned within three days. This makes roughly three days a reasonable starting point for testing a return reminder, but the result only describes customers who eventually reached three visits and does not establish the best timing for everyone.


In [4]:
second_order_days = repeat_customers["days_to_second_order"]
second_order_summary = pl.DataFrame({
    "Customers": [repeat_customers.height],
    "Median days": [second_order_days.median()],
    "Average days": [second_order_days.mean()],
    "Shortest wait (days)": [second_order_days.min()],
    "Longest wait (days)": [second_order_days.max()],
    "Within 3 days (%)": [100 * (second_order_days <= 3).sum() / repeat_customers.height],
}).with_columns(pl.col(pl.Float64).round(2))
display(second_order_summary)
second_order_plot = (
    ggplot(repeat_customers, aes(x="days_to_second_order"))
    + geom_histogram(binwidth=1, boundary=0, fill=blue, color="white")
    + geom_vline(xintercept=second_order_days.median(), color=green, linetype="dashed", size=1.1, manual_key="Median: 2.98 days")
    + scale_x_continuous(breaks=list(range(0, 15, 2)))
    + labs(title="Repeat customers usually placed a second order within a few days", subtitle="105 customers who reached 3+ visits; each bar covers one day", x="Days from first to second order", y="Customers")
    + theme_minimal()
    + theme(legend_position="bottom")
    + ggsize(850, 430)
)
display(second_order_plot)


Customers,Median days,Average days,Shortest wait (days),Longest wait (days),Within 3 days (%)
105,2.98,3.20,0.79,13.15,53.33


## Do repeat customers prefer certain product types, and do they spend more per order?

Beverages made up 83.7% of the items bought by repeat customers, compared with 51.3% for one-visit customers and 60.3% for two-visit customers. Repeat customers' baskets were more drink-focused, although they still bought fewer drinks per order: 1.14 versus 2.00 and 1.69. The five drinks had similar shares within the repeat group, so there is little reason to favor one flavor. These are differences in buying patterns, not evidence that purchasing a drink causes another visit.


In [5]:
quantity_columns = [name for name in orders.columns if name.endswith("_qty")]
order_items = (
    orders.unpivot(
        on=quantity_columns,
        index=["order_id", "customer_visit_group"],
        variable_name="quantity_column",
        value_name="units",
    )
    .with_columns(
        pl.col("quantity_column").str.replace("_qty", "").str.replace("_", "-").str.to_uppercase().alias("sku")
    )
    .join(products.select("sku", "product_name", "product_type"), on="sku")
)
orders_by_group = orders.group_by("customer_visit_group").agg(pl.len().alias("orders"))
product_types = (
    order_items.group_by("customer_visit_group", "product_type")
    .agg(pl.col("units").sum())
    .join(orders_by_group, on="customer_visit_group")
    .with_columns(
        (100 * pl.col("units") / pl.col("units").sum().over("customer_visit_group")).alias("unit_share"),
        (pl.col("units") / pl.col("orders")).alias("units_per_order"),
    )
    .sort("customer_visit_group", "product_type")
)
display(product_types.select(
    pl.col("customer_visit_group").alias("Visits"),
    pl.col("product_type").alias("Product type"),
    pl.col("orders").alias("Orders"),
    pl.col("units").alias("Units"),
    pl.col("unit_share").round(1).alias("Share of items (%)"),
    pl.col("units_per_order").round(2).alias("Units per order"),
))
product_type_plot = (
    ggplot(product_types, aes(x="customer_visit_group", y="unit_share", fill="product_type"))
    + geom_bar(stat="identity", width=0.6)
    + geom_text(aes(label="unit_share", group="product_type"), label_format="{.1f}%", position=position_stack(vjust=0.5), color="white", size=12)
    + scale_fill_manual(values=[blue, green], breaks=["beverage", "jaffle"], labels=["Beverages", "Jaffles"])
    + scale_x_discrete(limits=group_order, labels=group_labels)
    + scale_y_continuous(limits=[0, 100], breaks=[0, 25, 50, 75, 100])
    + labs(title="Repeat customers' baskets were more drink-focused", subtitle="Share of purchased units; 39, 73, and 885 items in the three groups", x="", y="Purchased items (%)", fill="Product type")
    + theme_minimal()
    + theme(legend_position="bottom")
    + ggsize(850, 430)
)
display(product_type_plot)

product_shares = (
    order_items.group_by("customer_visit_group", "sku", "product_name", "product_type")
    .agg(pl.col("units").sum())
    .with_columns((100 * pl.col("units") / pl.col("units").sum().over("customer_visit_group")).alias("share"))
    .pivot(on="customer_visit_group", index=["sku", "product_name", "product_type"], values="share")
    .sort("sku")
    .select(
        pl.col("product_name").alias("Product"),
        pl.col("product_type").alias("Type"),
        pl.col("1").round(1).alias("1 visit (%)"),
        pl.col("2").round(1).alias("2 visits (%)"),
        pl.col("3+").round(1).alias("3+ visits (%)"),
    )
)
display(product_shares)


Visits,Product type,Orders,Units,Share of items (%),Units per order
"""1""","""beverage""",10,20,51.30,2.00
"""1""","""jaffle""",10,19,48.70,1.90
"""2""","""beverage""",26,44,60.30,1.69
"""2""","""jaffle""",26,29,39.70,1.12
"""3+""","""beverage""",650,741,83.70,1.14
"""3+""","""jaffle""",650,144,16.30,0.22


Product,Type,1 visit (%),2 visits (%),3+ visits (%)
"""tangaroo""","""beverage""",2.60,9.60,16.50
"""chai and mighty""","""beverage""",12.80,13.70,16.20
"""vanilla ice""","""beverage""",7.70,4.10,17.40
"""for richer or pourover""","""beverage""",12.80,15.10,17.20
"""adele-ade""","""beverage""",15.40,17.80,16.50
"""nutellaphone who dis?""","""jaffle""",7.70,5.50,3.40
"""doctor stew""","""jaffle""",10.30,11.00,2.80
"""the krautback""","""jaffle""",5.10,9.60,3.80
"""flame impala""","""jaffle""",12.80,5.50,3.40
"""mel-bun""","""jaffle""",12.80,8.20,2.80


Repeat customers spent **less per order**: an average of $9.61, compared with $35.93 for one-visit customers and $23.77 for two-visit customers. Combining the one- and two-visit groups gives $27.15 per order, still well above the repeat group. The one- and two-visit groups also had higher median order totals, so the difference is not only a few expensive orders. Repeat customers produced more total sales because they ordered more often; this suggests testing a smaller return purchase rather than assuming a larger basket creates loyalty.


In [6]:
spending = (
    orders.group_by("customer_visit_group")
    .agg(
        pl.col("customer_id").n_unique().alias("customers"),
        pl.len().alias("orders"),
        pl.col("order_total_dollars").mean().alias("average_total"),
        pl.col("order_total_dollars").median().alias("median_total"),
        pl.col("item_count").mean().alias("items_per_order"),
    )
    .sort("customer_visit_group")
    .with_columns(pl.col("average_total").round(2).map_elements(lambda value: f"${value:.2f}", return_dtype=pl.String).alias("label"))
)
display(spending.select(
    pl.col("customer_visit_group").alias("Visits"),
    pl.col("customers").alias("Customers"),
    pl.col("orders").alias("Orders"),
    pl.col("average_total").round(2).alias("Average order ($)"),
    pl.col("median_total").round(2).alias("Median order ($)"),
    pl.col("items_per_order").round(2).alias("Items per order"),
))
combined_nonrepeat_average = orders.filter(~pl.col("is_repeat_customer"))["order_total_dollars"].mean()
display(pl.DataFrame({
    "Comparison": ["1-2 visits combined", "3+ visits"],
    "Customers": [active_customers.height - repeat_customers.height, repeat_customers.height],
    "Orders": [orders.height - repeat_orders.height, repeat_orders.height],
    "Average order ($)": [round(combined_nonrepeat_average, 2), round(repeat_orders["order_total_dollars"].mean(), 2)],
}))
spending_plot = (
    ggplot(spending, aes(x="customer_visit_group", y="average_total", fill="customer_visit_group"))
    + geom_bar(stat="identity", width=0.6, show_legend=False)
    + geom_text(aes(label="label"), nudge_y=1.5, size=12, show_legend=False)
    + scale_fill_manual(values=group_colors, breaks=group_order)
    + scale_x_discrete(limits=group_order, labels=group_labels)
    + scale_y_continuous(limits=[0, 42], breaks=[0, 10, 20, 30, 40], format="${.0f}")
    + labs(title="More visits did not mean a larger order", subtitle="Average per order, including tax; 10, 26, and 650 orders in the three groups", x="", y="Average order total ($)")
    + theme_minimal()
    + ggsize(850, 430)
)
display(spending_plot)


Visits,Customers,Orders,Average order ($),Median order ($),Items per order
"""1""",10,10,35.93,32.86,3.90
"""2""",13,26,23.77,13.78,2.81
"""3+""",105,650,9.61,6.36,1.36


Comparison,Customers,Orders,Average order ($)
"""1-2 visits combined""",23,36,27.15
"""3+ visits""",105,650,9.61


## How often do repeat customers visit?

For each repeat customer, I calculated the average time between their consecutive orders. The median of these customer averages was 2.32 days, and the mean was 2.70 days, suggesting a typical pattern of returning every two to three days during this sample. Each customer gets equal weight in this summary. A reminder around that time is worth testing, but sixteen days of data cannot establish a lasting routine.


In [7]:
customer_intervals = repeat_customers["avg_days_between_visits"]
frequency_summary = pl.DataFrame({
    "Customers": [repeat_customers.height],
    "Median customer interval (days)": [customer_intervals.median()],
    "Mean customer interval (days)": [customer_intervals.mean()],
    "Shortest customer average (days)": [customer_intervals.min()],
    "Longest customer average (days)": [customer_intervals.max()],
}).with_columns(pl.col(pl.Float64).round(2))
display(frequency_summary)
frequency_plot = (
    ggplot(repeat_customers, aes(x="avg_days_between_visits"))
    + geom_histogram(binwidth=0.5, boundary=0, fill=blue, color="white")
    + geom_vline(xintercept=customer_intervals.median(), color=green, linetype="dashed", size=1.1, manual_key="Median customer interval: 2.32 days")
    + scale_x_continuous(breaks=list(range(0, 9)))
    + labs(title="A typical repeat customer returned every two to three days", subtitle="105 customers; one average interval per customer; half-day bins", x="Customer's average days between visits", y="Customers")
    + theme_minimal()
    + theme(legend_position="bottom")
    + ggsize(850, 430)
)
display(frequency_plot)


Customers,Median customer interval (days),Mean customer interval (days),Shortest customer average (days),Longest customer average (days)
105,2.32,2.70,1.15,7.01


## Do regulars at different store locations have different habits?

We cannot compare locations with this sample because all 686 orders came from Philadelphia. The other five stores opened in 2017-2019, after the September 2016 sample, so their zero orders do not indicate poor sales. A useful comparison would need orders from several operating stores over the same period and would need to define repeat visits within each store.


In [8]:
store_orders = orders.group_by("store_id").agg(
    pl.len().alias("orders"),
    pl.col("customer_id").n_unique().alias("customers"),
)
store_coverage = (
    stores.join(store_orders, on="store_id", how="left")
    .with_columns(pl.col("orders", "customers").fill_null(0))
    .with_columns(
        pl.when(pl.col("opened_at").dt.date() > orders["order_date"].max())
        .then(pl.lit("Opened after sample"))
        .otherwise(pl.lit("Open during sample"))
        .alias("coverage")
    )
    .sort("opened_at")
    .select(
        pl.col("store_name").alias("Store"),
        pl.col("opened_at").dt.date().alias("Opened"),
        pl.col("orders").alias("Orders"),
        pl.col("customers").alias("Customers"),
        pl.col("coverage").alias("Sample coverage"),
    )
)
display(store_coverage)


Store,Opened,Orders,Customers,Sample coverage
"""Philadelphia""",2016-09-01,686,128,"""Open during sample"""
"""Brooklyn""",2017-03-12,0,0,"""Opened after sample"""
"""Chicago""",2018-04-29,0,0,"""Opened after sample"""
"""San Francisco""",2018-05-09,0,0,"""Opened after sample"""
"""New Orleans""",2019-03-10,0,0,"""Opened after sample"""
"""Los Angeles""",2019-09-13,0,0,"""Opened after sample"""


# Actionable next steps for Jaffle Shop

Encourage customers to build a habit of coming in for drinks. On a customer's first visit in a month, give them a coupon for one free drink on their next visit. After they redeem it, give them another coupon for one free drink with the purchase of any jaffle on a later visit. This gives them a reason to return twice and fits the drink-focused habits in our data, where beverages made up 83.7% of repeat customers' purchased items.

| Visit | Offer | Goal |
|---|---|---|
| First visit in a month | Give a coupon for one free drink on the next visit | Encourage a second visit |
| Second visit | Redeem the free drink, then receive a coupon for a free drink with any jaffle on a later visit | Encourage a third visit |
| Third visit | Buy any jaffle and redeem the second coupon for a free drink | Encourage a paid meal and a habit of returning |

For new customers, a [Forbes Business Council article by Joseph Edgar](https://www.forbes.com/councils/forbesbusinesscouncil/2026/07/17/winning-customers-why-repeat-business-is-the-real-key-to-restaurant-success/) cites Jon Taffer's estimate that the chance of another visit is about 40%-42% after the second visit and over 70% after the third. This supports focusing on that third visit, but these are outside estimates, not results measured in our sample or a guarantee that this offer will work.

Test the coupon sequence with a group of customers and compare their second, third, and fourth visits with a similar group that does not receive it. Track paid purchases and the cost of the free drinks to check whether the extra visits are worthwhile.
